# 📐 Linear Algebra for Data Science
### eMobilis · Data Science & AI

This notebook covers linear algebra from scratch — assuming you know Python and basic stats, but nothing about vectors or matrices.

Every section follows this pattern:
1. **What problem does this solve?** — the real-world motivation
2. **Explanation** — in plain English, with worked examples
3. **Visualisation** — so you can *see* what's happening
4. **Code** — bringing it all together

---

| Module | Topic |
|--------|-------|
| 0 | Why We Need Linear Algebra (Bridge from Stats) |
| 1 | Vectors |
| 2 | Matrices |
| 3 | The Determinant |
| 4 | Matrix Inverse |
| 5 | Solving Linear Systems |
| 6 | Rank and Null Space |
| 7 | Eigenvalues and Eigenvectors |
| 8 | Matrix Decompositions |
| 9 | Real-World ML Pipelines |

In [ ]:
# Run this first — all imports used throughout
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from scipy import linalg
from scipy.linalg import null_space
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics.pairwise import cosine_similarity

# Make all plots look clean and readable
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f8f9fa',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.size':        11,
})

print("All imports ready ✓")

---
# Module 0 — Why We Need Linear Algebra

## The problem: statistics runs out of road

In the statistics module you learned tools that work really well — but they all share one limitation: they handle **one or two variables at a time**.

- A distribution describes **one** variable (e.g. how are daily trips distributed?)
- A t-test compares **one** value between two groups
- Covariance and Pearson r describe the relationship between **two** variables

That is fine for simple questions. But real data looks like this:

| matatu | trips | fare (K KES) | fuel (L) | passengers | distance (km) | breakdowns | driver_exp | age_yrs | route | … |
|--------|-------|-------------|----------|------------|--------------|------------|------------|---------|-------|---|
| 1      | 8     | 12.0        | 3.5      | 120        | 25           | 0          | 5          | 3       | CBD   | … |
| 2      | 10    | 15.0        | 4.1      | 145        | 30           | 1          | 8          | 1       | Thika | … |
| …      | …     | …           | …        | …          | …            | …          | …          | …       | …     | … |

A real sacco dataset might have **50 features** per matatu and **200 matatus**. That is 10,000 numbers. If you want to compute Pearson r for every possible pair of features, you need **1,225 separate correlation calculations** just for 50 features (500 features → 124,750 calculations). You cannot do that one at a time.

**Linear algebra is the mathematical language that handles all variables simultaneously.** Instead of computing one thing at a time, you set up one matrix operation and get answers for everything at once.

## You already used linear algebra without knowing it

Every stats concept you learned has a linear algebra version underneath it:

| Stats concept (what you know) | Linear algebra version (what it really is) |
|---|---|
| One matatu's data row | A **vector** |
| Your whole fleet CSV | A **matrix** |
| Cov(X, Y) from last week | One single cell in a **covariance matrix** that holds all covariances at once |
| Pearson r = Cov/(σXσY) | The **cosine similarity** (normalised dot product) of two vectors |
| Finding the regression line | Solving a **matrix equation** AX = B |
| PCA directions | The **eigenvectors** of the covariance matrix |

The difference is not what you compute — it is that linear algebra computes all of it in one step, and does it **orders of magnitude faster** because modern CPUs and GPUs are literally built to do matrix operations.

In [ ]:
# =================================================================
# How much faster is NumPy vs a Python for-loop?
# Let's measure it concretely.\n# =================================================================
import time, numpy as np

np.random.seed(42)
n = 100_000   # 100,000 matatus — a real fleet simulation

trips      = np.random.randint(5, 20, size=n)
fare_rates = np.random.uniform(1.5, 3.0, size=n)

# --- Method 1: Python for-loop (the natural but slow way) ---
start = time.perf_counter()
revenues_loop = []
for i in range(n):
    revenues_loop.append(trips[i] * fare_rates[i])
loop_ms = (time.perf_counter() - start) * 1000

# --- Method 2: NumPy (one C-level call that does all 100k at once) ---
start = time.perf_counter()
revenues_np = trips * fare_rates
np_ms = (time.perf_counter() - start) * 1000

print(f"100,000 matatu revenue calculations:")
print(f"  For-loop : {loop_ms:.1f} ms")
print(f"  NumPy    : {np_ms:.3f} ms")
print(f"  Speedup  : {loop_ms/np_ms:.0f}x faster")
print(f"  Same result? {np.allclose(revenues_loop, revenues_np)}")
print()
print("Why is NumPy faster?")
print("  1. Python loops check types and allocate memory PER element.")
print("  2. NumPy sends the entire array to compiled C code in ONE call.")
print("  3. Modern CPUs can multiply 4-8 numbers in a SINGLE instruction (SIMD).")
print("  4. Data is stored contiguously in memory so the CPU cache works perfectly.")

---
# Module 1 — Vectors

## 1.1 What Is a Vector?

**The simplest definition:** A vector is an ordered list of numbers.

The word "ordered" is important — it means each position in the list has a specific meaning. Look at this matatu record:

```
Position →   0        1         2         3
           [trips,  fare_k,  fuel_L,  passengers]
matatu_1 = [  8,    12.0,     3.5,      120     ]
```

Position 0 always means trips. Position 1 always means fare. If you shuffle the numbers, the meaning breaks. That is why the order matters.

**A vector = one row of your dataset.**

Every time you have one observation in your CSV — one matatu, one student, one transaction — that row is a vector. All the numbers describing that single thing, collected together in order.

**The dimension of a vector** is simply how many numbers it contains. Count them.

```
[5]                      →  1-dimensional (1D): one number, lives on a number line
[3, 4]                   →  2-dimensional (2D): two numbers, like a point on a map
[8, 12.0, 3.5]           →  3-dimensional (3D): three numbers, a point in 3D space
[8, 12.0, 3.5, 120]      →  4-dimensional (4D): four numbers — can't draw it, but numpy handles it fine
[v₁, v₂, ..., v₅₀₀]     →  500D: one row of a 500-feature dataset
```

> **Important:** When we say "500-dimensional," we do NOT mean physical space. It just means 500 numbers packed in a list. Your brain can't visualise it — that's OK, numpy doesn't care about that. It computes in any number of dimensions the same way.

**Row vector vs Column vector**

The same numbers can be written horizontally (row) or vertically (column). They are the same data — orientation only matters when you start multiplying matrices (we will come back to this).

```
Row vector:    v = [3,  4,  5]        shape in numpy: (3,)

Column vector: v = [ 3 ]              shape in numpy: (3, 1)
                   [ 4 ]
                   [ 5 ]
```

In data science, datasets store each sample as a row — so most of the time you will work with row vectors. Just know that column vectors exist.

In [ ]:
import numpy as np

# =================================================================
# What a vector looks like in numpy
# =================================================================

# One matatu's daily data = one vector
matatu_1 = np.array([8, 12.0, 3.5, 120])   # [trips, fare_k, fuel_L, passengers]
matatu_2 = np.array([10, 15.0, 4.1, 145])  # a different matatu

print("matatu_1 :", matatu_1)
print("Dimension :", len(matatu_1), "  ← just count the numbers")
print("Shape     :", matatu_1.shape, "  ← numpy's way of saying 4-dimensional")

# Each position has a meaning
print("\nAccessing individual components:")
print("  trips      (index 0):", matatu_1[0])
print("  fare       (index 1):", matatu_1[1])
print("  fuel       (index 2):", matatu_1[2])
print("  passengers (index 3):", matatu_1[3])

# Row vs column vector — same data, different shape
row = np.array([3.0, 4.0, 5.0])           # shape (3,)
col = row.reshape(-1, 1)                   # shape (3, 1)

print("\nRow vector shape:", row.shape)
print("Col vector shape:", col.shape)
print("They contain the same numbers — orientation differs")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =================================================================
# VISUALISATION: What does a 2D vector look like?
# We will use trips vs fare as our two dimensions.
# =================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left plot: vectors as arrows from the origin ---
ax = axes[0]
vectors = {
    'matatu 1 [8, 12]':    ([8, 12],  'royalblue'),
    'matatu 2 [10, 15]':   ([10, 15], 'crimson'),
    'matatu 3 [4, 7]':     ([4, 7],   'seagreen'),
}

for label, (v, color) in vectors.items():
    ax.annotate('', xy=(v[0], v[1]), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    ax.text(v[0]+0.2, v[1]+0.2, label, color=color, fontsize=9, fontweight='bold')
    ax.plot(*v, 'o', color=color, markersize=8)

ax.set_xlim(-1, 13); ax.set_ylim(-1, 17)
ax.axhline(0, color='black', lw=0.8); ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Trips (dimension 0)', fontsize=11)
ax.set_ylabel('Fare — K KES (dimension 1)', fontsize=11)
ax.set_title('Each matatu is a point / arrow in 2D space', fontsize=12, fontweight='bold')

# --- Right plot: same vectors shown as a data table ---
ax2 = axes[1]
ax2.axis('off')
data = [['matatu 1', '8',  '12.0'],
        ['matatu 2', '10', '15.0'],
        ['matatu 3', '4',  '7.0']]
headers = ['', 'trips (dim 0)', 'fare_k (dim 1)']
table = ax2.table(cellText=data, colLabels=headers, cellLoc='center', loc='center')
table.scale(1.4, 2.2)
table.auto_set_font_size(False); table.set_fontsize(12)
for (r, c), cell in table.get_celld().items():
    if r == 0:
        cell.set_facecolor('#1F3864'); cell.set_text_props(color='white', fontweight='bold')
    elif c == 0:
        cell.set_facecolor('#EAF2FB')
    cell.set_edgecolor('#CCCCCC')
ax2.set_title('The same data as a table — each row is a vector', fontsize=12, fontweight='bold', pad=20)

plt.tight_layout()
plt.suptitle('A vector is a point in space AND a row in your dataset', fontsize=13, y=1.02, fontweight='bold')
plt.show()
print("Each arrow starts at the origin (0,0) and ends at the vector's coordinates.")
print("The length and direction of the arrow together represent the vector.")

## 1.2 Vector Operations

### Part A — Addition: Combining Two Observations

**What it is:** Add the matching numbers in each position.

Position 0 adds to position 0. Position 1 adds to position 1. Every position adds independently.

**Concrete example — a matatu's morning + afternoon runs:**

```
Morning  : [6 trips,  9k fare]
Afternoon: [4 trips,  7k fare]
                ↓        ↓
Day total: [6+4,     9+7    ] = [10 trips, 16k fare]
```

You can think of this geometrically too: if the first vector is a movement "go 6 right and 9 up", and the second is "go 4 right and 7 up", adding them gives you the combined movement "go 10 right and 16 up".

**Rule:** Both vectors must have the **same dimension** (same number of components). You cannot add a 3D vector to a 4D vector — there is no matching position for the extra component.

---

### Part B — Scalar Multiplication: Scaling Everything

**What it is:** Multiply every number in the vector by the same single number (called a **scalar**).

```
Vector: v = [3, 4]

Multiply by 2:   2 × [3, 4] = [6, 8]    ← same direction, twice as long
Multiply by 0.5: 0.5 × [3, 4] = [1.5, 2] ← same direction, half as long
Multiply by -1:  -1 × [3, 4] = [-3, -4]  ← REVERSED direction, same length
```

**Real-world:** If fare prices double across the whole fleet, you multiply the fare component of every matatu's vector by 2.

---

### Part C — Magnitude: How Long Is the Vector?

The **magnitude** of a vector (written ||v||) is its length — the straight-line distance from the origin to the tip of the arrow.

In 2D, this is just Pythagoras' theorem:

```
v = [3, 4]

Draw the triangle:
         ↑ 4
         |      ← this side = 4
         |  /
         | /  ← this is the vector v, length = ?
         |/___
              3   ← this side = 3

Length = √(3² + 4²) = √(9 + 16) = √25 = 5
```

In 3D: length = √(v₁² + v₂² + v₃²)

In nD: length = √(v₁² + v₂² + ... + vₙ²)

It is the same formula extended — just more terms under the square root.

**Why it matters in ML:** If matatu A has a feature vector with magnitude 5 and matatu B has magnitude 500, their scales are completely different. Before comparing or training models, we need to bring them to the same scale. That is what normalisation does.

---

### Part D — Unit Vectors: Pure Direction, No Scale

**The problem:** When comparing two matatus, you might only care about their *pattern* (do they run similar routes?) not their *scale* (one is a big bus, one is a matatu).

**What a unit vector is:** It points in exactly the same direction as the original vector, but its length has been shrunk to exactly 1.

Here is what that looks like:

```
Original:  v = [3, 4]    length = 5

              ↑ 4
              |  /
              | /   ← length 5
              |/
              ----→ 3

Unit:      û = [0.6, 0.8]   length = 1

              ↑ 0.8
              | /
              |/  ← same direction, but length 1
              -→ 0.6
```

**How to compute it:** Divide every component by the total length.

```
v = [3, 4],  length = 5

û = v / length = [3/5, 4/5] = [0.6, 0.8]

Check: √(0.6² + 0.8²) = √(0.36 + 0.64) = √1 = 1  ✓
```

This process is called **normalisation**.

**Why it matters in ML:**
- Before computing similarity between feature vectors, you normalise first so that "how big" a vector is doesn't drown out "what pattern" it represents
- Neural network weight initialisation often uses unit vectors
- Cosine similarity (coming next) is built entirely on this idea

In [ ]:
import numpy as np

# =================================================================
# 1.2A  Vector addition
# =================================================================
morning   = np.array([6, 9.0])    # [trips, fare_k] — morning shift
afternoon = np.array([4, 7.0])    # afternoon shift

day_total = morning + afternoon   # numpy adds position-by-position
print("Morning  :", morning)
print("Afternoon:", afternoon)
print("Day total:", day_total)    # [10. 16.]
print("  trips: 6+4 =", morning[0]+afternoon[0])
print("  fare : 9+7 =", morning[1]+afternoon[1])

# =================================================================
# 1.2B  Scalar multiplication
# =================================================================
v = np.array([3.0, 4.0])
print("\nOriginal v:", v)
print("v × 2     :", v * 2)          # stretched
print("v × 0.5   :", v * 0.5)        # shrunk
print("v × (-1)  :", v * -1)         # reversed direction

# =================================================================
# 1.2C  Magnitude
# =================================================================
manual = np.sqrt(v[0]**2 + v[1]**2)  # Pythagoras by hand
numpy_way = np.linalg.norm(v)         # numpy does the same thing
print("\nMagnitude of [3, 4]:")
print("  Manual (√(3²+4²)) :", manual)
print("  np.linalg.norm(v) :", numpy_way)
print("  Both equal 5?     :", np.isclose(manual, numpy_way))

# Higher dimensions — same formula, just more terms
v3 = np.array([3.0, 4.0, 0.0])
v4 = np.array([1.0, 2.0, 2.0, 0.0])
print("\n||[3,4,0]||  =", np.linalg.norm(v3))   # still 5 (0 contributes nothing)
print("||[1,2,2,0]|| =", np.linalg.norm(v4))     # √(1+4+4) = 3

# =================================================================
# 1.2D  Unit vector (normalisation)
# =================================================================
v = np.array([3.0, 4.0])
length = np.linalg.norm(v)       # step 1: compute length (= 5)
unit_v = v / length              # step 2: divide every component by length

print("\nUnit vector:")
print("  Original v     :", v,     "  length =", length)
print("  unit_v = v/5   :", unit_v, "  length =", np.linalg.norm(unit_v))
print("  Check: same direction?", np.allclose(unit_v / np.linalg.norm(unit_v), unit_v))
print("  Check: length = 1?", np.isclose(np.linalg.norm(unit_v), 1.0))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =================================================================
# VISUALISATION: Vector operations — addition, scaling, unit vector
# =================================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ---- Plot 1: Vector Addition (tip-to-tail method) ----
ax = axes[0]
u = np.array([6.0, 9.0])   # morning
v = np.array([4.0, 7.0])   # afternoon
w = u + v                   # day total

def draw_arrow(ax, start, end, color, label='', lw=2.5, ls='-'):
    ax.annotate('', xy=end, xytext=start,
                arrowprops=dict(arrowstyle='->', color=color, lw=lw, linestyle=ls))
    mid = (start + end) / 2
    if label:
        ax.text(mid[0]+0.3, mid[1]+0.3, label, color=color, fontsize=9.5, fontweight='bold')

draw_arrow(ax, np.array([0,0]), u, 'royalblue', 'morning [6,9]')
draw_arrow(ax, u, u+np.array([4,7]), 'seagreen', 'afternoon [4,7]')
draw_arrow(ax, np.array([0,0]), w, 'crimson', 'total [10,16]', lw=3)

ax.set_xlim(-1, 13); ax.set_ylim(-1, 19)
ax.axhline(0, color='black', lw=0.8); ax.axvline(0, color='black', lw=0.8)
ax.set_title('Vector Addition\nTip-to-tail: morning → then afternoon', fontweight='bold')
ax.set_xlabel('trips'); ax.set_ylabel('fare_k')

# ---- Plot 2: Scalar Multiplication ----
ax = axes[1]
v = np.array([3.0, 4.0])
scales = [(2.0, 'royalblue', '× 2  →  [6, 8]'),
          (1.0, 'black',     '× 1  →  [3, 4]  (original)'),
          (0.5, 'seagreen',  '× 0.5 → [1.5, 2]'),
          (-1.0,'crimson',   '× (-1) → [-3,-4] (reversed)')]

for s, col, lbl in scales:
    sv = v * s
    draw_arrow(ax, np.array([0,0]), sv, col, lbl, lw=2.5 if s != 1 else 3)

ax.set_xlim(-5, 8); ax.set_ylim(-6, 10)
ax.axhline(0, color='black', lw=0.8); ax.axvline(0, color='black', lw=0.8)
ax.set_title('Scalar Multiplication\nSame direction, different length', fontweight='bold')
ax.set_xlabel('component 0'); ax.set_ylabel('component 1')

# ---- Plot 3: Unit Vector ----
ax = axes[2]
v = np.array([3.0, 4.0])
u_v = v / np.linalg.norm(v)

draw_arrow(ax, np.array([0,0]), v,   'royalblue', f'original [3,4]\nlength={np.linalg.norm(v):.0f}', lw=3)
draw_arrow(ax, np.array([0,0]), u_v, 'crimson',   f'unit [0.6,0.8]\nlength=1.0', lw=2.5)

# Show the right angle for magnitude
ax.plot([3,3],[0,4], color='royalblue', lw=1.5, ls='--', alpha=0.6)
ax.plot([0,3],[0,0], color='royalblue', lw=1.5, ls='--', alpha=0.6)
ax.text(1.5, -0.4, '3', color='royalblue', ha='center', fontsize=11)
ax.text(3.3, 2.0,  '4', color='royalblue', ha='left',   fontsize=11)
ax.text(1.0, 2.5,  'length = √(3²+4²) = 5', color='royalblue', fontsize=9, style='italic')

ax.set_xlim(-0.5, 4.5); ax.set_ylim(-0.8, 5.0)
ax.axhline(0, color='black', lw=0.8); ax.axvline(0, color='black', lw=0.8)
ax.set_title('Unit Vector\nSame direction, length shrunk to 1', fontweight='bold')
ax.set_xlabel('component 0'); ax.set_ylabel('component 1')

plt.tight_layout()
plt.show()

## 1.3 The Dot Product — The Single Most Important Operation in ML

If you had to pick just one operation to understand deeply, this is it. The dot product is used inside every ML model you will ever build: linear regression, neural networks, recommendation systems, NLP, image search.

### What the dot product computes

Take two vectors of the same dimension. Multiply the matching components together, then add all those products. The result is a single number (called a **scalar**).

**Step by step:**

```
u = [1, 2, 3]
v = [4, 5, 6]

Step 1 — multiply matching components:
  position 0:  1 × 4 = 4
  position 1:  2 × 5 = 10
  position 2:  3 × 6 = 18

Step 2 — add them all together:
  4 + 10 + 18 = 32

dot product = 32
```

That is it. One number out. The formula written compactly: **u · v = u₁v₁ + u₂v₂ + ... + uₙvₙ**

### What does that number mean?

The number tells you **how much the two vectors agree in direction**.

Imagine two vectors as arrows drawn from the same starting point. The angle between them tells you how "aligned" they are:

```
CASE 1: Arrows point the same direction (angle = 0°)
    u →→→→
    v →→→→
    They agree completely. Dot product is LARGE and POSITIVE.

CASE 2: Arrows are perpendicular (angle = 90°)
    u →→→→
    v ↑
      ↑
    They share no common direction at all. Dot product = 0.

CASE 3: Arrows point opposite directions (angle = 180°)
    u →→→→
    v ←←←←
    They completely disagree. Dot product is LARGE and NEGATIVE.
```

The formula that captures this: **u · v = ||u|| × ||v|| × cos(θ)**

Where θ is the angle between u and v, and cos(θ) goes from +1 (same direction) to 0 (perpendicular) to -1 (opposite).

### Why this matters in every ML model

Every time a model "scores" an input, it uses a dot product:

```
features = [trips=10, passengers=145, fare=15]
weights  = [0.5,      0.02,           1.0   ]   ← learned by the model

score = 10×0.5 + 145×0.02 + 15×1.0
      = 5.0   + 2.9        + 15.0
      = 22.9

This score is the model's prediction for that matatu.
```

This is exactly what `LinearRegression.predict()` does for every single sample.

In [ ]:
import numpy as np

# =================================================================
# 1.3  THE DOT PRODUCT
# =================================================================

# Step-by-step for u = [1,2,3] and v = [4,5,6]
u = np.array([1.0, 2.0, 3.0])
v = np.array([4.0, 5.0, 6.0])

products = u * v           # multiply matching components
print("Matching products:", products)    # [4, 10, 18]
total = products.sum()
print("Sum of products  :", total)       # 32
print("np.dot(u, v)     :", np.dot(u, v))  # same result

# --- Three classic cases ---
same = np.array([1.0, 1.0]);  v_same = np.array([2.0, 2.0])   # same direction
opp  = np.array([1.0, 1.0]);  v_opp  = np.array([-2.0,-2.0])  # opposite
perp = np.array([1.0, 0.0]);  v_perp = np.array([0.0, 1.0])   # 90 degrees

print("\n=== Three classic cases ===")
print(f"Same direction  [1,1]·[2,2]   = {np.dot(same, v_same):>5.1f}  ← large positive")
print(f"Opposite dir    [1,1]·[-2,-2] = {np.dot(opp, v_opp):>5.1f}  ← large negative")
print(f"Perpendicular   [1,0]·[0,1]   = {np.dot(perp, v_perp):>5.1f}  ← exactly zero")

# --- Real-world: scoring a matatu for revenue prediction ---
features = np.array([10.0, 145.0, 15.0])   # [trips, passengers, fare_k]
weights  = np.array([0.5,  0.02,  1.0])    # learned weights

print("\n=== Matatu revenue score (dot product) ===")
print(f"  trips × 0.5        = {features[0]*weights[0]:.1f}")
print(f"  passengers × 0.02  = {features[1]*weights[1]:.1f}")
print(f"  fare × 1.0         = {features[2]*weights[2]:.1f}")
print(f"  Total score        = {np.dot(features, weights):.1f}")
print("  This is what LinearRegression.predict() does for every sample")

# --- The angle formula ---
angle_deg = 45
angle_rad = np.radians(angle_deg)
u2 = np.array([1.0, 0.0])
v2 = np.array([np.cos(angle_rad), np.sin(angle_rad)])
dot = np.dot(u2, v2)
expected = np.linalg.norm(u2) * np.linalg.norm(v2) * np.cos(angle_rad)
print(f"\nFor 45° angle: u·v = {dot:.4f},  ||u||·||v||·cos(45°) = {expected:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =================================================================
# VISUALISATION: The dot product — three cases and the angle
# =================================================================

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

def draw_vec(ax, v, color, label, lw=2.5):
    ax.annotate('', xy=v, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw))
    ax.text(v[0]+0.1, v[1]+0.1, label, color=color, fontsize=10, fontweight='bold')

titles = ['Same direction\n[1,1]·[2,2] = 4 (positive)',
          'Perpendicular\n[1,0]·[0,1] = 0',
          'Opposite\n[1,1]·[-2,-2] = -4 (negative)']

cases = [
    (np.array([1.0,1.0]), np.array([2.0,2.0]),   'royalblue', 'seagreen'),
    (np.array([1.0,0.0]), np.array([0.0,1.0]),   'royalblue', 'seagreen'),
    (np.array([1.0,1.0]), np.array([-2.0,-2.0]), 'royalblue', 'crimson'),
]

for i, (u, v, cu, cv) in enumerate(cases):
    ax = axes[i]
    draw_vec(ax, u, cu, 'u')
    draw_vec(ax, v, cv, 'v')
    dot_val = np.dot(u, v)
    ax.set_xlim(-2.8, 2.8); ax.set_ylim(-2.8, 2.8)
    ax.axhline(0, color='black', lw=0.8); ax.axvline(0, color='black', lw=0.8)
    ax.set_title(titles[i], fontweight='bold', fontsize=10)
    color_bg = '#eafaf1' if dot_val > 0 else ('#fdecea' if dot_val < 0 else '#f8f9fa')
    ax.set_facecolor(color_bg)

# ---- Plot 4: The angle connection ----
ax = axes[3]
angles = np.linspace(0, np.pi, 300)
cos_vals = np.cos(angles)
ax.plot(np.degrees(angles), cos_vals, 'royalblue', lw=2.5)
ax.axhline(0, color='black', lw=1)
ax.fill_between(np.degrees(angles), cos_vals, 0,
                where=(cos_vals > 0), color='seagreen', alpha=0.2, label='Positive dot product')
ax.fill_between(np.degrees(angles), cos_vals, 0,
                where=(cos_vals < 0), color='crimson', alpha=0.2, label='Negative dot product')
ax.set_xlabel('Angle θ between vectors (degrees)')
ax.set_ylabel('cos(θ)')
ax.set_title('Dot product sign depends on angle\nu·v = ||u||·||v||·cos(θ)', fontweight='bold', fontsize=10)
ax.legend(fontsize=9)
ax.set_xticks([0,45,90,135,180])
ax.text(10, 0.5, 'Same dir\n→ positive', color='seagreen', fontsize=9)
ax.text(95, -0.5, 'Opposite\n→ negative', color='crimson', fontsize=9)

plt.suptitle('The Dot Product: measuring agreement between two vectors', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 1.4 Cosine Similarity — Comparing Patterns Without Being Fooled by Scale

### The problem with raw dot products

Imagine you want to find matatus with similar operating patterns. You compute the dot product between matatu A and matatu B.

```
matatu_A = [170 trips, 12000 fare_KES, 120 passengers]
matatu_B = [175 trips, 12500 fare_KES, 125 passengers]   ← very similar to A
matatu_C = [8   trips,  3000 fare_KES,  40 passengers]   ← very different from A
```

When you compute raw dot products:
- A · B ≈ 2,600,000,000 (huge — dominated by fare in KES)
- A · C ≈ 140,000 (also large, but should be low because C is different)

The fare column (with values in thousands of KES) completely dominates the result. You cannot compare these numbers meaningfully.

### The fix: normalise first

If you divide each vector by its own length first, you turn them into unit vectors — all with length 1. Then the dot product only measures **direction** (pattern), not size.

```
Step 1: Compute lengths
  ||A|| = √(170² + 12000² + 120²) ≈ 12001.3
  ||B|| ≈ 12502.4
  ||C|| ≈ 3001.7

Step 2: Divide by length (normalise)
  Â = A / ||A||   (unit vector pointing same direction as A)
  B̂ = B / ||B||
  Ĉ = C / ||C||

Step 3: Dot product of unit vectors = cosine similarity
  Â · B̂ ≈ 0.9999   ← A and B are almost identical in pattern ✓
  Â · Ĉ ≈ 0.9876   ← still similar? (because fare dominates even in ratios)
```

The formula written together:

**cosine_similarity(u, v) = (u · v) / (||u|| × ||v||)**

The result is always between -1 and +1:
- **+1** = identical pattern (same direction)
- **0** = completely unrelated patterns (perpendicular)
- **-1** = opposite patterns

**This is exactly Pearson r in disguise.** When you computed r = Cov(X,Y)/(σX × σY) in the stats module, you were computing the cosine similarity of the deviation vectors (X − X̄) and (Y − Ȳ). The formula is structurally identical.

### Where cosine similarity is used

- **NLP:** word embeddings — "king" and "queen" should have high cosine similarity
- **Recommendation systems:** user-item similarity — find users with similar taste patterns
- **Search engines:** match a query to documents based on direction of word frequency vectors
- **Fraud detection:** find transactions with similar feature patterns

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# =================================================================
# 1.4  COSINE SIMILARITY
# =================================================================

# Three matatus — compare their operating patterns
A = np.array([170.0, 12000.0, 120.0])   # [trips, fare_KES, passengers]
B = np.array([175.0, 12500.0, 125.0])   # very similar to A
C = np.array([  8.0,  3000.0,  40.0])   # very different from A

# --- Raw dot product: misleading ---
print("=== Raw dot products ===")
print(f"  A · B = {np.dot(A, B):>15,.0f}  (A and B are similar)")
print(f"  A · C = {np.dot(A, C):>15,.0f}  (A and C are different)")
print("  These numbers don't tell us much — dominated by fare (large units)")

# --- Manual cosine similarity ---
def cos_sim(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

print("\n=== Cosine similarity (normalised) ===")
sim_AB = cos_sim(A, B)
sim_AC = cos_sim(A, C)
print(f"  cosine_sim(A, B) = {sim_AB:.6f}  ← close to 1 (very similar) ✓")
print(f"  cosine_sim(A, C) = {sim_AC:.6f}  ← lower (less similar) ✓")

# --- sklearn shortcut ---
X = np.array([A, B, C])
sim_matrix = cosine_similarity(X)
print("\n=== Full 3×3 similarity matrix (sklearn) ===")
labels = ['A', 'B', 'C']
print(f"     {'A':>9}  {'B':>9}  {'C':>9}")
for i, row in enumerate(sim_matrix):
    print(f"  {labels[i]}  {row[0]:>9.4f}  {row[1]:>9.4f}  {row[2]:>9.4f}")

# --- Connection to Pearson r ---
x = np.array([8.0, 10, 12, 14, 16])
y = np.array([12.0, 15, 16, 20, 22])

# Pearson r from last module:
r_pearson = np.corrcoef(x, y)[0, 1]

# Cosine similarity of deviation vectors:
x_dev = x - x.mean()
y_dev = y - y.mean()
r_cosine = cos_sim(x_dev, y_dev)

print(f"\n=== Pearson r IS cosine similarity ===")
print(f"  Pearson r (np.corrcoef)           : {r_pearson:.6f}")
print(f"  Cosine sim of deviation vectors   : {r_cosine:.6f}")
print(f"  They are the same?                : {np.isclose(r_pearson, r_cosine)}")
print("  r = Cov(X,Y)/(σX·σY) is exactly the cosine similarity of (X-X̄) and (Y-Ȳ)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =================================================================
# VISUALISATION: Why normalisation matters for similarity
# =================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Two 2D matatus — same PATTERN but different SCALE
A2 = np.array([3.0, 4.0])    # small matatu
B2 = np.array([9.0, 12.0])   # same route, 3x busier (scaled version)
C2 = np.array([4.0, 1.0])    # genuinely different route

# ---- Left: raw vectors ----
ax = axes[0]
colors = {'A [3,4]': ('royalblue', A2),
          'B [9,12] (3×A)': ('seagreen', B2),
          'C [4,1] (different)': ('crimson', C2)}

for label, (col, v) in colors.items():
    ax.annotate('', xy=v, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color=col, lw=2.5))
    ax.text(v[0]+0.2, v[1]+0.2, label, color=col, fontsize=9.5, fontweight='bold')

raw_AB = np.dot(A2, B2)
raw_AC = np.dot(A2, C2)
ax.set_xlim(-1, 11); ax.set_ylim(-1, 14)
ax.axhline(0, color='black', lw=0.8); ax.axvline(0, color='black', lw=0.8)
ax.set_title(f'Raw dot products\nA·B={raw_AB:.0f}  vs  A·C={raw_AC:.0f}  ← misleading!',
             fontweight='bold')
ax.set_xlabel('component 0'); ax.set_ylabel('component 1')

# ---- Right: unit vectors ----
ax = axes[1]
for label, (col, v) in colors.items():
    u = v / np.linalg.norm(v)
    ax.annotate('', xy=u, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color=col, lw=2.5))
    ax.text(u[0]+0.03, u[1]+0.03, label, color=col, fontsize=9.5, fontweight='bold')

Au = A2/np.linalg.norm(A2); Bu = B2/np.linalg.norm(B2); Cu = C2/np.linalg.norm(C2)
cos_AB = np.dot(Au, Bu)
cos_AC = np.dot(Au, Cu)

theta = np.linspace(0, 2*np.pi, 200)
ax.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3, lw=1.5)   # unit circle
ax.set_xlim(-1.4, 1.6); ax.set_ylim(-1.2, 1.4)
ax.axhline(0, color='black', lw=0.8); ax.axvline(0, color='black', lw=0.8)
ax.set_title(f'After normalising (unit vectors)\ncosine_sim(A,B)={cos_AB:.4f}  vs  cosine_sim(A,C)={cos_AC:.4f} ← now makes sense!',
             fontweight='bold')
ax.set_xlabel('component 0 (normalised)'); ax.set_ylabel('component 1 (normalised)')
ax.text(0.0, -1.1, '← unit circle (all lengths = 1)', ha='center', fontsize=9, style='italic', color='grey')

plt.suptitle('Cosine Similarity: compare patterns, not sizes', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("A and B have the SAME pattern (B is just 3× bigger) → cosine similarity ≈ 1.0")
print("A and C have different patterns → cosine similarity is lower")

## 1.5 Linear Independence — Does Each Vector Add New Information?

### The core question

If you have a collection of vectors, can you build any one of them from the others?

- If **yes** → that vector is **redundant** — it tells you nothing you couldn't already figure out
- If **no** → that vector is genuinely **independent** — it points in a new direction

### A concrete example

Imagine tracking three matatus along two dimensions: trips (x) and fare (y).

```
matatu A: v_A = [4, 8]   →  4 trips, 8k fare
matatu B: v_B = [2, 4]   →  2 trips, 4k fare
matatu C: v_C = [3, 7]   →  3 trips, 7k fare
```

Can you build v_B from v_A?

```
v_B = 0.5 × v_A = 0.5 × [4, 8] = [2, 4]  ✓
```

Yes! v_B is just v_A scaled by 0.5. If you already have v_A, you can recover v_B exactly by multiplying by 0.5. v_B adds **no new directional information**. They point in the same direction — just different lengths.

Can you build v_C from v_A?

```
Try: k × [4, 8] = [3, 7]
  From position 0: k = 3/4 = 0.75
  Check position 1: 0.75 × 8 = 6.0  ≠  7
```

No — no single scaling makes both components match. v_C points in a genuinely **new direction**. It is independent.

### What "linearly independent" means formally

Vectors are linearly independent if the **only** way to combine them to get zero is to multiply all of them by zero.

If you can find any non-zero combination that gives zero, some of them are dependent.

### Why this matters in machine learning

This is where it gets practical. If two columns of your dataset are linearly dependent (perfectly correlated), you have a problem:

1. Your dataset grows in size but gains **no new information**
2. The matrix your model needs to invert becomes **singular** — you get a "Singular matrix" error
3. Even if it doesn't crash, coefficient estimates become **meaningless and unstable**

Common examples of dependent columns you might accidentally include:
- `fare_KES` and `fare_USD` (one is just the other × exchange rate)
- `fuel_litres` and `fuel_millilitres` (one is 1000 × the other)
- `total_trips` and `morning_trips + afternoon_trips` (exact linear combination)

In [ ]:
import numpy as np

# =================================================================
# 1.5  LINEAR INDEPENDENCE
# =================================================================

v_A = np.array([4.0, 8.0])
v_B = np.array([2.0, 4.0])   # = 0.5 × v_A  →  DEPENDENT
v_C = np.array([3.0, 7.0])   # cannot be scaled from v_A  →  INDEPENDENT

# Check if v_B is a scalar multiple of v_A
# If the ratios are the same in every position, they point the same direction
ratio_B = v_B / v_A
print("v_B / v_A =", ratio_B)   # [0.5, 0.5] — same ratio everywhere → DEPENDENT

ratio_C = v_C / v_A
print("v_C / v_A =", ratio_C)   # [0.75, 0.875] — different ratios → INDEPENDENT

# --- Practical: a dataset with dependent columns ---
trips     = np.array([8.0, 10, 12, 14, 16])
fare_k    = np.array([12.0, 15, 16, 20, 22])          # fare in K KES
fare_ksh  = fare_k * 1000                              # exact same info, different unit
dist_km   = np.array([25.0, 30, 33, 40, 45])          # genuinely new info

# Stack them as columns
X_bad  = np.column_stack([trips, fare_k, fare_ksh, dist_km])
X_good = np.column_stack([trips, fare_k, dist_km])

print(f"\nBad dataset  (with redundant fare_ksh): rank = {np.linalg.matrix_rank(X_bad)}")
print(f"Good dataset (fare_ksh removed)         : rank = {np.linalg.matrix_rank(X_good)}")
print(f"(Bad dataset has 4 columns but only rank 3 → one column is redundant)")

# --- What happens when you try to use dependent features ---
print("\nAttempting to invert X_bad.T @ X_bad ...")
try:
    inv = np.linalg.inv(X_bad.T @ X_bad)
    print("Inverse computed (unexpected)")
except np.linalg.LinAlgError as e:
    print(f"  Error: {e}")
    print("  Cause: fare_ksh = 1000 × fare_k  →  linear dependence  →  singular matrix")
    print("  Fix: remove fare_ksh before fitting any model")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =================================================================
# VISUALISATION: Linearly dependent vs independent vectors
# =================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ---- Left: dependent vectors ----
ax = axes[0]
vA = np.array([4.0, 8.0])
vB = np.array([2.0, 4.0])   # = 0.5 × vA

ax.annotate('', xy=vA, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='royalblue', lw=3))
ax.annotate('', xy=vB, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='crimson',   lw=2.5, linestyle='--'))
ax.text(vA[0]+0.2, vA[1]+0.2, 'v_A = [4, 8]', color='royalblue', fontsize=11, fontweight='bold')
ax.text(vB[0]+0.2, vB[1]+0.2, 'v_B = [2, 4]\n= 0.5 × v_A', color='crimson', fontsize=11, fontweight='bold')

# Draw the line they both lie on
t = np.linspace(-0.5, 1.2, 100)
ax.plot(t * vA[0], t * vA[1], 'purple', lw=1.5, ls=':', alpha=0.7, label='same line')
ax.text(3.5, 0.5, '← both vectors lie\non the SAME line', color='purple', fontsize=9.5, style='italic')

ax.set_xlim(-1, 6); ax.set_ylim(-1, 10)
ax.axhline(0, color='black', lw=0.8); ax.axvline(0, color='black', lw=0.8)
ax.set_title('DEPENDENT vectors\nThey point the same direction — one is redundant', fontweight='bold')
ax.set_xlabel('trips'); ax.set_ylabel('fare')
ax.legend()

# ---- Right: independent vectors ----
ax = axes[1]
vA = np.array([4.0, 8.0])
vC = np.array([3.0, 7.0])

ax.annotate('', xy=vA, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='royalblue', lw=3))
ax.annotate('', xy=vC, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='seagreen',  lw=2.5))
ax.text(vA[0]+0.2, vA[1]+0.1, 'v_A = [4, 8]', color='royalblue', fontsize=11, fontweight='bold')
ax.text(vC[0]-2.5, vC[1]+0.1, 'v_C = [3, 7]\n(different direction)', color='seagreen', fontsize=11, fontweight='bold')

# Show the angle between them
import matplotlib.patches as patches
angle_A = np.degrees(np.arctan2(vA[1], vA[0]))
angle_C = np.degrees(np.arctan2(vC[1], vC[0]))
arc = patches.Arc((0,0), 3, 3, angle=0, theta1=min(angle_A,angle_C), theta2=max(angle_A,angle_C),
                   color='orange', lw=2)
ax.add_patch(arc)
ax.text(0.8, 2.5, f'angle ≠ 0°', color='orange', fontsize=10, fontweight='bold')

ax.set_xlim(-1, 6); ax.set_ylim(-1, 10)
ax.axhline(0, color='black', lw=0.8); ax.axvline(0, color='black', lw=0.8)
ax.set_title('INDEPENDENT vectors\nDifferent directions — each adds new information', fontweight='bold')
ax.set_xlabel('trips'); ax.set_ylabel('fare')

plt.suptitle('Linear Independence: does each vector point in a new direction?',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
# Module 2 — Matrices: Your Entire Dataset as One Object

## 2.1 What Is a Matrix?

A matrix is a **rectangular grid of numbers arranged in rows and columns**. It has a name like "5 × 4 matrix" — that means 5 rows and 4 columns (rows first, always).

But the most important way to understand a matrix in data science is this:

**A matrix IS your dataset.**

```
          trips  fare_k  fuel_L  passengers
          ↓      ↓       ↓       ↓
matatu 1 [  8,   12.0,   3.5,    120   ]   ← row 0
matatu 2 [ 10,   15.0,   4.1,    145   ]   ← row 1
matatu 3 [ 12,   16.0,   4.4,    160   ]   ← row 2
matatu 4 [ 14,   20.0,   5.0,    180   ]   ← row 3
matatu 5 [ 16,   22.0,   5.3,    200   ]   ← row 4
```

This 5 × 4 matrix contains ALL the data for 5 matatus across 4 features. Every `pd.read_csv()` call creates a matrix. A pandas DataFrame IS a matrix with labels on top.

**Reading elements:** We identify each cell by (row index, column index).
```
a[2, 1] = 16.0   ← row 2, column 1 → matatu 3's fare
a[0, 0] = 8      ← row 0, column 0 → matatu 1's trips
a[4, 3] = 200    ← row 4, column 3 → matatu 5's passengers
```

**Why store your data as a matrix?**

1. **Memory efficiency:** All 20 numbers live in one contiguous block of memory. The CPU can load whole chunks into its fast cache at once. Scattered Python lists cause "cache misses" that slow everything down.

2. **Speed:** Operations like "compute the mean of every column" or "standardise all features" are done on the whole matrix in one call at the C level — not one row at a time in Python.

3. **Mathematical power:** Matrix operations let you express complex multi-variable calculations (like linear regression for 200 matatus at once) in a single concise formula.

In [ ]:
import numpy as np
import pandas as pd

# =================================================================
# 2.1  THE MATRIX AS YOUR DATASET
# =================================================================

# Our fleet: 5 matatus × 4 features
fleet = np.array([
    [8,  12.0, 3.5, 120],   # matatu 1
    [10, 15.0, 4.1, 145],   # matatu 2
    [12, 16.0, 4.4, 160],   # matatu 3
    [14, 20.0, 5.0, 180],   # matatu 4
    [16, 22.0, 5.3, 200],   # matatu 5
])

print("=== The matrix ===")
print(fleet)
print(f"\nShape: {fleet.shape}  ← (rows=matatus, cols=features)")
print(f"This is a {fleet.shape[0]}×{fleet.shape[1]} matrix")

print("\n=== Accessing elements [row, col] ===")
print(f"fleet[0, 0] = {fleet[0,0]}   ← row 0, col 0  → matatu 1's trips")
print(f"fleet[2, 1] = {fleet[2,1]}  ← row 2, col 1  → matatu 3's fare")
print(f"fleet[4, 3] = {fleet[4,3]}  ← row 4, col 3  → matatu 5's passengers")

print("\n=== Slicing rows and columns ===")
print("All trips (column 0)   :", fleet[:, 0])
print("Matatu 3 (row 2)       :", fleet[2])
print("Top 3 matatus (rows 0-2):")
print(fleet[:3])

# The DataFrame is the same matrix with labels
df = pd.DataFrame(fleet, columns=["trips","fare_k","fuel_L","passengers"])
print("\n=== As a DataFrame (same matrix, with labels) ===")
print(df)
print("\nType of df.values:", type(df.values), "← it's a numpy array underneath")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =================================================================
# VISUALISATION: A matrix as a colour grid (heatmap)
# This is how the computer "sees" your dataset
# =================================================================

fleet = np.array([
    [8,  12.0, 3.5, 120],
    [10, 15.0, 4.1, 145],
    [12, 16.0, 4.4, 160],
    [14, 20.0, 5.0, 180],
    [16, 22.0, 5.3, 200],
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- Left: the numbers as a grid ----
ax = axes[0]
im = ax.imshow(fleet, cmap='Blues', aspect='auto')

col_names = ['trips', 'fare_k', 'fuel_L', 'passengers']
row_names = [f'matatu {i+1}' for i in range(5)]

ax.set_xticks(range(4)); ax.set_xticklabels(col_names, fontsize=11)
ax.set_yticks(range(5)); ax.set_yticklabels(row_names, fontsize=11)
ax.set_xlabel('Features (columns →)', fontsize=11)
ax.set_ylabel('Observations (rows ↓)', fontsize=11)
ax.set_title(f'Fleet matrix — {fleet.shape[0]}×{fleet.shape[1]}\n(colour intensity = value size)', fontweight='bold')
plt.colorbar(im, ax=ax)

# Annotate each cell with its value
for i in range(5):
    for j in range(4):
        ax.text(j, i, fleet[i, j], ha='center', va='center',
                color='white' if fleet[i,j] > fleet[:,j].mean() else 'black',
                fontsize=10, fontweight='bold')

# ---- Right: highlight specific cells ----
ax2 = axes[1]
highlight = np.zeros_like(fleet)
highlight[2, 1] = 1   # highlight matatu 3's fare\n
im2 = ax2.imshow(fleet, cmap='Blues', aspect='auto', alpha=0.4)
ax2.imshow(highlight, cmap='Reds', aspect='auto', alpha=0.6)

ax2.set_xticks(range(4)); ax2.set_xticklabels(col_names, fontsize=11)
ax2.set_yticks(range(5)); ax2.set_yticklabels(row_names, fontsize=11)
ax2.set_title(f'Accessing fleet[2, 1] = {fleet[2,1]}\n(matatu 3, fare column — highlighted in red)', fontweight='bold')

for i in range(5):
    for j in range(4):
        ax2.text(j, i, fleet[i, j], ha='center', va='center', fontsize=10, fontweight='bold')

plt.suptitle('A matrix = your dataset as a grid of numbers', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 2.4 Matrix Multiplication — The Engine of All Machine Learning

This is the most important operation in all of machine learning. Once you understand it, every ML algorithm becomes easier to follow.

**Matrix multiplication is NOT multiplying matching elements.** It follows a specific rule.

### The rule

To find element (i, j) in the result, take **row i of A** and **column j of B**, multiply matching components, and sum them up. That is a dot product.

**Worked example:**

```
A = [[1, 2],      B = [[5, 6],
     [3, 4]]           [7, 8]]

Result C = A @ B:

C[0,0] = row 0 of A  ·  col 0 of B
       = [1, 2]  ·  [5, 7]
       = 1×5 + 2×7
       = 5 + 14 = 19

C[0,1] = row 0 of A  ·  col 1 of B
       = [1, 2]  ·  [6, 8]
       = 1×6 + 2×8 = 22

C[1,0] = row 1 of A  ·  col 0 of B
       = [3, 4]  ·  [5, 7]
       = 3×5 + 4×7 = 43

C[1,1] = row 1 of A  ·  col 1 of B
       = [3, 4]  ·  [6, 8]
       = 3×6 + 4×8 = 50

C = [[19, 22],
     [43, 50]]
```

### The dimension rule

For A @ B to work, **the number of columns of A must equal the number of rows of B**.

```
A is (m × n)   and   B is (n × p)   →   result is (m × p)

The n gets "consumed". m and p survive.

Example:
  (200 × 5) @ (5 × 1)  =  (200 × 1)
   200 matatus  5 weights  200 predictions
```

This is how linear regression makes predictions for ALL matatus at once — not one at a time.

### Element-wise multiply vs matrix multiply — do NOT confuse these

```
A * B  (star)  →  multiply matching positions  →  ELEMENT-WISE  →  quick scaling
A @ B  (at)    →  row × column dot products   →  MATRIX MULTIPLY →  ML predictions, transformations
```

They give completely different results.

In [ ]:
import numpy as np

# =================================================================
# 2.4  MATRIX MULTIPLICATION
# =================================================================

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

# --- Step by step ---
print("=== Manual calculation ===")
print("C[0,0] = row0(A) · col0(B) =", A[0],"·",[B[0,0],B[1,0]], "= 1×5+2×7 =", 1*5+2*7)
print("C[0,1] = row0(A) · col1(B) =", A[0],"·",[B[0,1],B[1,1]], "= 1×6+2×8 =", 1*6+2*8)
print("C[1,0] = row1(A) · col0(B) =", A[1],"·",[B[0,0],B[1,0]], "= 3×5+4×7 =", 3*5+4*7)
print("C[1,1] = row1(A) · col1(B) =", A[1],"·",[B[0,1],B[1,1]], "= 3×6+4×8 =", 3*6+4*8)
print("\nA @ B =")
print(A @ B)

# --- CRITICAL: element-wise vs matrix multiply ---
print("\n=== Element-wise (A * B) — NOT matrix multiply ===")
print(A * B)   # [[5,12],[21,32]]  ← completely different!

print("\n=== Matrix multiply (A @ B) ===")
print(A @ B)   # [[19,22],[43,50]]

# --- The real power: predict for ALL matatus in ONE operation ---
fleet = np.array([
    [1, 8,  12.0, 3.5, 120],   # bias=1, trips, fare_k, fuel_L, passengers
    [1, 10, 15.0, 4.1, 145],
    [1, 12, 16.0, 4.4, 160],
    [1, 14, 20.0, 5.0, 180],
    [1, 16, 22.0, 5.3, 200],
])   # shape (5, 5)

weights = np.array([50.0, 2.0, 1.5, -10.0, 0.3])   # shape (5,)

# Shape check: (5×5) @ (5,) → (5,) — one prediction per matatu
predictions = fleet @ weights
print("\n=== Predictions for 5 matatus — ONE matrix multiply ===")
print(f"fleet shape  : {fleet.shape}")
print(f"weights shape: {weights.shape}")
print(f"result shape : {predictions.shape}  ← one value per matatu")
print("Predictions:", predictions.round(1))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =================================================================
# VISUALISATION: Matrix multiplication as row × column dot products
# =================================================================

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
C = A @ B

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

def draw_matrix(ax, M, title, highlight_row=None, highlight_col=None, cmap='Blues'):
    im = ax.imshow(M, cmap=cmap, vmin=0, vmax=M.max()*1.2, aspect='auto')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            color = 'white' if M[i,j] > M.max()*0.5 else 'black'
            ax.text(j, i, f'{M[i,j]:.0f}', ha='center', va='center',
                    color=color, fontsize=16, fontweight='bold')
    if highlight_row is not None:
        ax.add_patch(plt.Rectangle((-0.5, highlight_row-0.5), M.shape[1], 1,
                                   fill=True, color='gold', alpha=0.4, lw=0))
    if highlight_col is not None:
        ax.add_patch(plt.Rectangle((highlight_col-0.5, -0.5), 1, M.shape[0],
                                   fill=True, color='tomato', alpha=0.4, lw=0))
    ax.set_xticks(range(M.shape[1])); ax.set_yticks(range(M.shape[0]))
    ax.set_title(title, fontweight='bold', fontsize=12)
    return im

# Show which row and column produce C[0,0] = 19
draw_matrix(axes[0], A, 'A\n(highlight row 0)', highlight_row=0, cmap='Blues')
draw_matrix(axes[1], B, 'B\n(highlight col 0)', highlight_col=0, cmap='Greens')
draw_matrix(axes[2], C, 'C = A @ B\n(C[0,0] = row0·col0 = 1×5+2×7 = 19)', highlight_row=0, highlight_col=0, cmap='Oranges')

axes[0].set_xlabel('A columns'); axes[0].set_ylabel('A rows')
axes[1].set_xlabel('B columns'); axes[1].set_ylabel('B rows')
axes[2].set_xlabel('C columns'); axes[2].set_ylabel('C rows')

plt.suptitle('Matrix multiply: each cell in C = dot product of one row of A with one column of B',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("C[0,0] = [1,2] · [5,7] = 1×5 + 2×7 = 19  ← highlighted cells above")
print("C[0,1] = [1,2] · [6,8] = 1×6 + 2×8 = 22")
print("C[1,0] = [3,4] · [5,7] = 3×5 + 4×7 = 43")
print("C[1,1] = [3,4] · [6,8] = 3×6 + 4×8 = 50")

## 2.5 Transpose — Flipping a Matrix on Its Diagonal

**What it does:** Swap rows and columns. The element at position (row i, column j) moves to (row j, column i).

```
Original A (2×3):             Transpose A.T (3×2):
[[1, 2, 3],                   [[1, 4],
 [4, 5, 6]]                    [2, 5],
                                [3, 6]]
```

Row 0 of A becomes column 0 of A.T. Row 1 of A becomes column 1 of A.T.

If A has shape (m × n), then A.T has shape (n × m).

**Why you see it in every ML formula:**

The linear regression formula is: **β = (XᵀX)⁻¹ Xᵀ y**

- X is (m × n) — m samples, n features
- Xᵀ is (n × m) — transposed
- Xᵀ X is (n × n) — SQUARE, so it can be inverted
- Xᵀ y is (n,) — one value per feature

Every time you see the ᵀ superscript in a formula, transpose is making the dimensions align so the multiplication is valid.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =================================================================
# 2.5  TRANSPOSE
# =================================================================

A = np.array([[1, 2, 3],
              [4, 5, 6]])   # shape (2, 3)

print("Original A  (2×3):")
print(A)
print("Shape:", A.shape)

print("\nA.T  (3×2):")
print(A.T)
print("Shape:", A.T.shape)

# Show element movement
print("\nElement at A[0,2] = 3  →  moves to  A.T[2,0] =", A.T[2,0])
print("Element at A[1,0] = 4  →  moves to  A.T[0,1] =", A.T[0,1])

# Shape trace through the regression formula
np.random.seed(0)
X = np.random.rand(100, 5)   # 100 matatus, 5 features
y = np.random.rand(100)

print("\n=== Shapes in regression formula β = (X.T @ X)^{-1} @ X.T @ y ===")
print(f"X         shape: {X.shape}")
print(f"X.T       shape: {X.T.shape}")
print(f"X.T @ X   shape: {(X.T @ X).shape}  ← SQUARE — can be inverted ✓")
print(f"X.T @ y   shape: {(X.T @ y).shape}  ← right-hand side")
print(f"beta      shape: {np.linalg.solve(X.T @ X, X.T @ y).shape}  ← one coeff per feature")

# Visualisation: before and after transpose
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
M = np.array([[1,2,3],[4,5,6]])
for ax, mat, title in zip(axes, [M, M.T], ['Original A  (2×3)', 'Transpose A.T  (3×2)']):
    im = ax.imshow(mat, cmap='Blues', aspect='auto', vmin=0, vmax=7)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, mat[i,j], ha='center', va='center',
                    color='white' if mat[i,j]>3 else 'black', fontsize=16, fontweight='bold')
    ax.set_xticks(range(mat.shape[1])); ax.set_yticks(range(mat.shape[0]))
    ax.set_xlabel('columns'); ax.set_ylabel('rows')
    ax.set_title(title, fontweight='bold')
    plt.colorbar(im, ax=ax)
plt.suptitle('Transpose: rows become columns, columns become rows', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 2.6 The Covariance Matrix — All Relationships at Once

In the stats module you computed Cov(trips, fare) for **one pair** of variables. That gives one number.

A real dataset might have 50 features. You would need to compute Cov for every pair — that is 1,225 separate calculations. The **covariance matrix** holds all of those in a single structure:

```
           trips      fare      fuel
trips   [ Var(trips)  Cov(t,f)  Cov(t,u) ]
fare    [ Cov(f,t)  Var(fare)   Cov(f,u) ]
fuel    [ Cov(u,t)  Cov(u,f)   Var(fuel) ]
```

- The **diagonal** contains the variance of each feature (how spread out it is)
- The **off-diagonal** cells contain the covariance between each pair (how they move together)
- The matrix is **symmetric** — Cov(trips, fare) = Cov(fare, trips)

`df.corr()` gives you the same matrix but standardised — all values between -1 and +1 (Pearson r). That is the correlation matrix, and every off-diagonal cell is the r value you computed by hand before.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =================================================================
# 2.6  COVARIANCE MATRIX
# =================================================================

fleet = np.array([[8,12.0,3.5],[10,15.0,4.1],[12,16.0,4.4],[14,20.0,5.0],[16,22.0,5.3]])
labels = ["trips", "fare_k", "fuel_L"]

# np.cov expects features as ROWS, so we transpose (each feature becomes one row)
C = np.cov(fleet.T)
print("Covariance matrix (3×3 — all feature pairs at once):")
df_cov = pd.DataFrame(C, index=labels, columns=labels)
print(df_cov.round(3))

print("\nDiagonal = variances:")
for i, lbl in enumerate(labels):
    print(f"  Var({lbl:10}) = {C[i,i]:.3f}")

print("\nOff-diagonal = covariances:")
print(f"  Cov(trips, fare)  = {C[0,1]:.3f}")
print(f"  Cov(trips, fuel)  = {C[0,2]:.3f}")

# Correlation matrix (standardised: all values in [-1, 1])
df = pd.DataFrame(fleet, columns=labels)
corr = df.corr()
print("\nCorrelation matrix (Pearson r for all pairs):")
print(corr.round(3))

# Visualise both
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, mat, title, fmt in zip(axes,
    [df_cov.values, corr.values],
    ['Covariance Matrix', 'Correlation Matrix (Pearson r)'],
    ['.1f', '.3f']):
    im = ax.imshow(mat, cmap='RdYlGn', vmin=-abs(mat).max(), vmax=abs(mat).max(), aspect='auto')
    ax.set_xticks(range(3)); ax.set_xticklabels(labels, fontsize=11)
    ax.set_yticks(range(3)); ax.set_yticklabels(labels, fontsize=11)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{mat[i,j]:{fmt}}', ha='center', va='center',
                    fontsize=12, fontweight='bold',
                    color='white' if abs(mat[i,j]) > abs(mat).max()*0.6 else 'black')
    plt.colorbar(im, ax=ax)
    ax.set_title(title, fontweight='bold')
plt.suptitle('Covariance and correlation matrices — all relationships in one grid',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()
print("Green = positive relationship. Red = negative. Diagonal = self-relationship (variance/1.0).")

---
# Module 3 — The Determinant

## What Is the Determinant?

The determinant is a **single number** you can compute from any square matrix. On its own it does not tell you much, but it answers one extremely important question:

**Can this matrix be reversed (inverted)?**

### The geometric meaning

Think of a 2×2 matrix as a machine that transforms space. You put vectors in, it spits out transformed vectors. The determinant tells you **by how much the area scales** when the transformation is applied.

```
Before transformation: you have a unit square (area = 1)

                 ↑
                 | (0,1)
                 |
      -----------+--------→
                (0,0)      (1,0)

After applying matrix A:
  det(A) = 2  → the square became a parallelogram with area 2 (doubled)
  det(A) = 1  → area unchanged (pure rotation)
  det(A) = 0  → the square COLLAPSED to a LINE (area = 0!)
```

**The key insight:** If `det(A) = 0`, the matrix destroyed information — it squashed 2D space into a line (or 3D space into a plane, etc.). You cannot get back the original information. The transformation cannot be reversed. No inverse exists.

### Computing it for a 2×2 matrix

```
A = [[a, b],
     [c, d]]

det(A) = a×d - b×c
```

Example:
```
A = [[7, 2],        det = 7×5 - 2×17 = 35 - 34 = 1    → invertible ✓
     [17, 5]]

B = [[2, 4],        det = 2×2 - 4×1 = 4 - 4 = 0        → SINGULAR ✗
     [1, 2]]
```

Look at B: row 1 = [2, 4] = 2 × [1, 2] = 2 × row 0. The rows are linearly dependent. The matrix flattens the plane — no inverse.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =================================================================
# MODULE 3  THE DETERMINANT
# =================================================================

# 2×2 manual formula: det = ad - bc
A = np.array([[7.0, 2.0], [17.0, 5.0]])
B = np.array([[2.0, 4.0], [1.0, 2.0]])   # singular

det_A = A[0,0]*A[1,1] - A[0,1]*A[1,0]
det_B = B[0,0]*B[1,1] - B[0,1]*B[1,0]

print(f"det(A) = 7×5 - 2×17 = {det_A}   → invertible ✓")
print(f"det(B) = 2×2 - 4×1  = {det_B}   → SINGULAR (row1 = 2×row0)")
print(f"numpy det(A) = {np.linalg.det(A):.1f}")
print(f"numpy det(B) = {np.linalg.det(B):.1f}")

# Practical check
print("\n=== Always check det before inverting ===")
for name, M in [("A (invertible)", A), ("B (singular)", B)]:
    det = np.linalg.det(M)
    print(f"  {name}: det={det:.1f}, can_invert={not np.isclose(det,0)}")

# Visualisation: how the unit square transforms
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

def show_transform(ax, M, title, det_val):
    unit_sq = np.array([[0,0],[1,0],[1,1],[0,1],[0,0]], dtype=float).T
    transformed = M @ unit_sq
    ax.fill(unit_sq[0], unit_sq[1], alpha=0.25, color='royalblue', label='unit square (area=1)')
    ax.fill(transformed[0], transformed[1], alpha=0.35, color='crimson',
            label=f'transformed (area={abs(det_val):.1f})')
    ax.plot(unit_sq[0], unit_sq[1], 'royalblue', lw=2)
    ax.plot(transformed[0], transformed[1], 'crimson', lw=2)
    for (xi,yi) in unit_sq.T[:-1]:
        p = M @ np.array([xi, yi])
        ax.annotate('', xy=p, xytext=(xi,yi),
                    arrowprops=dict(arrowstyle='->', color='grey', lw=1))
    ax.axhline(0, color='k',lw=0.7); ax.axvline(0, color='k',lw=0.7)
    ax.set_title(f'{title}\ndet = {det_val:.1f}', fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_aspect('equal')

M1 = np.array([[2.,0],[0,1]])
M2 = np.array([[1.,-0.5],[0.5,1]])
M3 = np.array([[2.,4],[1,2]])
for ax, M, title in zip(axes,
    [M1, M2, M3],
    ['[[2,0],[0,1]]\nStretches x2', '[[1,-0.5],[0.5,1]]\nRotates+shears', '[[2,4],[1,2]]\nSINGULAR']):
    show_transform(ax, M, title, np.linalg.det(M))
    lo, hi = -0.5, 5.5
    ax.set_xlim(lo,hi); ax.set_ylim(-1,hi)

plt.suptitle('The determinant = how much area the matrix transformation scales by',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()
print("det=0: the red shape collapsed to a LINE — information destroyed, no inverse possible")

---
# Module 4 — The Matrix Inverse

## What Is the Inverse?

For the number 5, its inverse is 1/5, because 5 × (1/5) = 1.

For a matrix A, its inverse A⁻¹ satisfies: **A × A⁻¹ = A⁻¹ × A = I**

Where I is the identity matrix (the "1" for matrices — has 1s on the diagonal and 0s everywhere else).

If A represents a transformation (rotating, stretching), then A⁻¹ represents the exact reverse transformation (un-rotating, un-stretching). The inverse **undoes** what A did.

### When does the inverse not exist?

1. **det(A) = 0** — the matrix collapsed space; you cannot undo that
2. **Matrix is not square** — a 3×2 matrix has no inverse (not even defined)

### The formula for 2×2 matrices

```
A = [[a, b],       A⁻¹ = (1/det) × [[ d, -b],
     [c, d]]                         [-c,  a]]

Steps:
  1. Swap the diagonal elements (a ↔ d)
  2. Negate the off-diagonal elements (b → -b, c → -c)
  3. Divide everything by det(A)
```

### 🚨 Never use inv() to solve a linear system

This is one of the most important practical lessons. If you want to solve AX = B, the formula X = A⁻¹B is mathematically correct, but in code you should NEVER compute the inverse explicitly.

**Why?**

- Computing A⁻¹ first and then multiplying takes more steps
- It accumulates more floating-point rounding errors
- For large matrices, it is noticeably slower

**Always use `np.linalg.solve(A, B)` instead.** It uses LU decomposition (covered in Module 8) internally, which is faster and more accurate.

In [ ]:
import numpy as np
import time

# =================================================================
# MODULE 4  MATRIX INVERSE
# =================================================================

A = np.array([[7.0, 2.0], [17.0, 5.0]])

# --- Manual 2×2 formula ---
a,b,c,d = A[0,0],A[0,1],A[1,0],A[1,1]
det = a*d - b*c
A_inv = (1/det) * np.array([[d,-b],[-c,a]])

print("Manual inverse:")
print(A_inv)

# Verify A @ A⁻¹ = I
print("\nA @ A_inv (should be identity matrix):")
print(np.round(A @ A_inv, 10))

# numpy's inv
A_inv_np = np.linalg.inv(A)
print("\nnumpy inv matches manual:", np.allclose(A_inv, A_inv_np))

# =================================================================
# CRITICAL: inv() vs solve() — speed and stability comparison
# =================================================================
b_vec = np.array([9.0, 3.0])

# BAD approach (textbook habit)
X_bad  = np.linalg.inv(A) @ b_vec
# GOOD approach (what you should always use)
X_good = np.linalg.solve(A, b_vec)

print("\n=== Solving AX = b ===")
print("inv() approach  :", X_bad)    # correct but wasteful
print("solve() approach:", X_good)   # same answer, better numerics

# Speed difference on a large matrix
np.random.seed(0)
N = 1000
big_A = np.random.rand(N,N) + N*np.eye(N)   # make sure it's invertible
big_b = np.random.rand(N)

t0 = time.perf_counter(); _ = np.linalg.inv(big_A) @ big_b; t_inv = time.perf_counter()-t0
t0 = time.perf_counter(); _ = np.linalg.solve(big_A, big_b); t_sol = time.perf_counter()-t0

print(f"\n1000×1000 system:")
print(f"  np.linalg.inv() @ b  : {t_inv*1000:.1f} ms")
print(f"  np.linalg.solve()    : {t_sol*1000:.1f} ms")
print(f"  solve() is {t_inv/t_sol:.1f}x faster (and more numerically stable)")
print("  Rule: always use solve() when you want to solve a system")

---
# Module 5 — Solving Linear Systems (AX = B)

## What Is a Linear System?

A **linear system** is a group of equations where every variable appears to the first power only — no x², no x×y, no sin(x). Just numbers multiplied by variables, added together.

```
2x + 3y = 9      ← equation 1
 x −  y = 3      ← equation 2
```

You want to find the values of x and y that satisfy BOTH equations simultaneously.

## Setting Up the Matrix Form

Instead of writing out x and y every time, we collect the numbers into a compact matrix form:

```
[ coefficients ] [ unknowns ] = [ results ]

[[2,  3],       [[x],        [[9],
 [1, -1]]    ×   [y]]     =   [3]]

    A                X     =    B
```

This lets you write the whole system as **AX = B**. Now instead of solving two equations, you are solving one matrix equation.

## Row Reduction — How to Solve It by Hand

You are allowed to do three things to the rows of the augmented matrix [A|B] without changing the solution:

1. **Swap rows** — reorder the equations
2. **Multiply a row by a non-zero number** — rescale an equation
3. **Add a multiple of one row to another** — combine equations to eliminate a variable

**Goal:** Use these operations to reach a form where the solution is obvious.

```
Start:         [[2,  3 | 9],
                [1, -1 | 3]]

Step 1: R0 ÷ 2  →  [[1, 1.5 | 4.5],   ← leading 1 in row 0
                     [1, -1  | 3  ]]

Step 2: R1 - R0 →  [[1,  1.5 | 4.5],
                     [0, -2.5 | -1.5]]  ← x is eliminated from row 1

Now read from row 1:  -2.5y = -1.5  →  y = 0.6
Substitute back:       x + 1.5(0.6) = 4.5  →  x = 3.6
```

**Row Echelon Form (REF):** The "staircase" pattern — zeros below each leading number. Solve by working backwards from the bottom.

**Reduced Row Echelon Form (RREF):** Every leading number is exactly 1, AND zeros above AND below each leading 1. Solution is readable directly from the last column — no substitution needed.

In [ ]:
import numpy as np

# =================================================================
# MODULE 5  SOLVING AX = B
# =================================================================

# The system: 2x + 3y = 9  and  x - y = 3

A = np.array([[2.0, 3.0], [1.0, -1.0]])
b = np.array([9.0, 3.0])

# --- Augmented matrix [A | b] ---
aug = np.hstack([A, b.reshape(-1,1)])
print("Augmented matrix [A | b]:")
print(aug)

# --- Step by step row reduction ---
aug[0] = aug[0] / 2           # R0 ← R0 / 2  →  leading element becomes 1
print("\nAfter R0 ÷ 2:")
print(aug)

aug[1] = aug[1] - aug[0]      # R1 ← R1 - R0  →  eliminate x from row 1
print("\nAfter R1 ← R1 - R0  (now in REF — staircase pattern):")
print(aug)

# Back-substitution from REF
y = aug[1,2] / aug[1,1]
x = (aug[0,2] - aug[0,1]*y) / aug[0,0]
print(f"\nBack-substitution: y = {y},  x = {x}")

# --- Continue to RREF (zeros above AND below pivots) ---
aug[1] /= aug[1,1]            # make pivot of row 1 exactly 1
aug[0] -= aug[0,1] * aug[1]   # eliminate above pivot

print("\nAfter continuing to RREF:")
print(aug)
print(f"Solution directly from last column: x = {aug[0,2]},  y = {aug[1,2]}")

# --- numpy does all of this in one call ---
solution = np.linalg.solve(A, b)
print(f"\nnp.linalg.solve: x = {solution[0]},  y = {solution[1]}")
print(f"Verify: A @ x = {A @ solution}  (should equal b = {b})")

# --- Special cases ---
print("\n=== Special Case: No solution ===")
A_no = np.array([[1.,1.],[1.,1.]]); b_no = np.array([3.,5.])
print(f"rank(A)={np.linalg.matrix_rank(A_no)}, rank([A|b])={np.linalg.matrix_rank(np.hstack([A_no,b_no.reshape(-1,1)]))}")
print("rank(A) < rank([A|b]) → equations contradict → no solution")

print("\n=== Special Case: Infinite solutions ===")
A_inf = np.array([[1.,2.],[2.,4.]]); b_inf = np.array([4.,8.])
print(f"rank(A)={np.linalg.matrix_rank(A_inf)}, rank([A|b])={np.linalg.matrix_rank(np.hstack([A_inf,b_inf.reshape(-1,1)]))}")
print("rank(A) = rank([A|b]) but rank < n_vars → free variable → infinite solutions")

---
# Module 7 — Eigenvalues and Eigenvectors: The Natural Directions

## The Core Idea

A matrix A is a transformation machine. You feed a vector in, and it comes out stretched, rotated, or flipped.

**Most vectors:** both their direction AND their length change when you apply A.

**Special vectors (eigenvectors):** ONLY their length changes. Their direction stays the same (or exactly reverses).

The formula: **A · v = λ · v**

- **v** is the eigenvector (the special vector whose direction survives)
- **λ** (lambda) is the eigenvalue (the number it gets stretched/shrunk by)

This says: applying the transformation A to v gives you back the same v, just scaled by λ.

### A visual example

```
Matrix A = [[2, 0],     ← scales x by 2, y by 3
            [0, 3]]

Take the vector v = [1, 0]:
  A × [1, 0] = [2, 0] = 2 × [1, 0]  ← same direction! scaled by 2.
  So [1, 0] is an eigenvector with eigenvalue 2.

Take the vector v = [0, 1]:
  A × [0, 1] = [0, 3] = 3 × [0, 1]  ← same direction! scaled by 3.
  So [0, 1] is an eigenvector with eigenvalue 3.

Take the vector v = [1, 1]:
  A × [1, 1] = [2, 3]  ← this is NOT the same direction as [1,1]
  So [1, 1] is NOT an eigenvector of this matrix.
```

The x and y axes are the "natural directions" of this transformation — everything along them just gets stretched. Everything else rotates too.

### Why eigenvalues matter in data science

| Application | What the eigenvectors/eigenvalues tell you |
|---|---|
| **PCA** | Eigenvectors of the covariance matrix = the directions of maximum variance. Eigenvalue = how much variance is in that direction. |
| **Google PageRank** | The webpage ranking vector IS the dominant eigenvector of the web's link matrix. |
| **Stability of models** | Eigenvalues of the gradient tell you if a training algorithm will converge or diverge. |
| **Graph analysis** | Eigenvalues of the adjacency matrix reveal community structure. |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =================================================================
# MODULE 7  EIGENVALUES AND EIGENVECTORS
# =================================================================

# Simple diagonal matrix: eigenvectors are the axes
A_diag = np.array([[2.0, 0.0],
                   [0.0, 3.0]])

vals, vecs = np.linalg.eig(A_diag)
print("Diagonal matrix A = [[2,0],[0,3]]:")
print(f"  Eigenvalues   : {vals}  ← x gets ×2, y gets ×3")
print(f"  Eigenvectors  : columns of\n{vecs}")
print("  (columns: [1,0] and [0,1] — the x and y axes)")

# More interesting matrix
A = np.array([[2.0, 1.0],
              [1.0, 2.0]])
eigenvalues, eigenvectors = np.linalg.eig(A)
print("\nMatrix A = [[2,1],[1,2]]:")
print(f"  Eigenvalues   : {eigenvalues}")
print(f"  Eigenvectors  :\n{eigenvectors.round(4)}")

# VERIFY: A @ v = λ × v
print("\n=== Verify: A @ v should equal λ × v ===")
for i in range(2):
    v   = eigenvectors[:, i]
    lam = eigenvalues[i]
    Av  = A @ v
    lv  = lam * v
    print(f"  λ={lam:.0f}: A@v={Av.round(4)},  λ×v={lv.round(4)},  equal={np.allclose(Av,lv)}")

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

def plot_transform(ax, M, title):
    colors = ['royalblue','crimson','seagreen','darkorange']
    test_vecs = [np.array([1.,0]), np.array([0.,1.]), np.array([1.,1.])/np.sqrt(2),
                 np.array([1.,-1.])/np.sqrt(2)]
    labels = ['[1,0]','[0,1]','[1,1]/√2','[1,-1]/√2']
    for v, lbl, col in zip(test_vecs, labels, colors):
        Mv = M @ v
        ax.annotate('', xy=v, xytext=(0,0), arrowprops=dict(arrowstyle='->', color=col, lw=2, alpha=0.4, linestyle='--'))
        ax.annotate('', xy=Mv, xytext=(0,0), arrowprops=dict(arrowstyle='->', color=col, lw=2.5))
        ax.text(Mv[0]+0.05, Mv[1]+0.05, f'A{lbl}', color=col, fontsize=8.5, fontweight='bold')
    ax.set_xlim(-1.5,3.5); ax.set_ylim(-2,3.5)
    ax.axhline(0, color='k',lw=0.8); ax.axvline(0, color='k',lw=0.8)
    ax.set_title(title, fontweight='bold')

plot_transform(axes[0], A_diag, 'Diagonal matrix [[2,0],[0,3]]\nDashed=before, Solid=after transformation')
plot_transform(axes[1], A, 'A=[[2,1],[1,2]]\nDashed=before, Solid=after')

vals_d, vecs_d = np.linalg.eig(A_diag)
for i,col in enumerate(['royalblue','crimson']):
    v = vecs_d[:,i]*2.5
    axes[0].annotate('', xy=v, xytext=(0,0),
                     arrowprops=dict(arrowstyle='->', color='purple', lw=3.5))
    axes[0].text(v[0]+0.1,v[1]+0.1,f'Eigenvec\nλ={vals_d[i]:.0f}',color='purple',fontsize=9,fontweight='bold')

vals2, vecs2 = np.linalg.eig(A)
for i in range(2):
    v = vecs2[:,i]*vals2[i]
    axes[1].annotate('', xy=v, xytext=(0,0),
                     arrowprops=dict(arrowstyle='->', color='purple', lw=3.5))
    axes[1].text(v[0]+0.1,v[1]+0.1,f'Eigenvec\nλ={vals2[i]:.0f}',color='purple',fontsize=9,fontweight='bold')

plt.suptitle('Eigenvectors (purple): the only directions where A only scales, never rotates',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()
print("Purple arrows: eigenvectors after transformation — they stay on their original line")
print("Coloured arrows: all other vectors change direction when A is applied")

## 7.3 PCA — Eigenvalues Applied: Finding the Most Important Directions

### What PCA does

Imagine your data is a cloud of points in a high-dimensional space. PCA finds the directions (axes) that capture the most spread in that cloud.

The direction with the maximum spread = **Principal Component 1**. The direction with the second-most spread (perpendicular to PC1) = **PC2**. And so on.

By keeping only the top k directions, you can represent your data in k dimensions instead of the original p dimensions — **dimensionality reduction**.

### How it works step by step

1. **Standardise:** Subtract the mean and divide by std for each feature. This ensures no feature dominates just because it has large numbers.

2. **Compute the covariance matrix:** `C = X.T @ X / (n-1)`. This is a p×p matrix. Its (i,j) entry is the covariance between feature i and feature j.

3. **Find eigenvectors of C:** The eigenvector with the largest eigenvalue points in the direction of maximum variance. The eigenvalue itself tells you how much variance is in that direction.

4. **Keep the top-k eigenvectors:** These are your k principal components. Together they form a matrix V_k.

5. **Project your data:** `X_reduced = X @ V_k`. Now every observation is described by k numbers instead of p.

The eigenvalue of each component tells you the fraction of total variance it captures — this is the "variance explained" percentage you see in PCA results.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# =================================================================
# 7.3  PCA STEP BY STEP WITH VISUALISATION
# =================================================================

# Create data with a clear main direction (correlated features)
np.random.seed(42)
n = 80
angle = np.pi / 4     # 45 degree main direction
direction = np.array([np.cos(angle), np.sin(angle)])
t = np.random.randn(n)
noise = np.random.randn(n, 2) * 0.3
data_raw = np.outer(t, direction) + noise

# Standardise
scaler = StandardScaler()
X = scaler.fit_transform(data_raw)

# Manual PCA
C = np.cov(X.T)
eigenvalues, eigenvectors = np.linalg.eigh(C)
idx = np.argsort(eigenvalues)[::-1]   # sort descending
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

var_explained = eigenvalues / eigenvalues.sum()

print("=== PCA Results ===")
print(f"Eigenvalues : {eigenvalues.round(4)}")
print(f"Variance explained: PC1={var_explained[0]*100:.1f}%  PC2={var_explained[1]*100:.1f}%")
print(f"Principal Component 1 direction: {eigenvectors[:,0].round(4)}")

# Project onto top-1 component (1D representation)
X_1d = X @ eigenvectors[:,:1]    # now each observation = 1 number

# sklearn
pca_sk = PCA(n_components=2)
X_sk   = pca_sk.fit_transform(X)
print(f"\nsklearn: PC1 explains {pca_sk.explained_variance_ratio_[0]*100:.1f}% of variance")

# Visualisation
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: original data
ax = axes[0]
ax.scatter(X[:,0], X[:,1], alpha=0.6, color='royalblue', s=40)
ax.set_title('Original data\n(standardised features)', fontweight='bold')
ax.set_xlabel('Feature 0'); ax.set_ylabel('Feature 1')
ax.set_aspect('equal')

# Middle: data with PC directions overlaid
ax = axes[1]
ax.scatter(X[:,0], X[:,1], alpha=0.6, color='royalblue', s=40)
scale = np.sqrt(eigenvalues)
for i, (col,lbl) in enumerate([('crimson','PC1 (most variance)'),('seagreen','PC2 (least variance)')]):
    v = eigenvectors[:,i] * scale[i] * 2
    ax.annotate('', xy=v, xytext=-v,
                arrowprops=dict(arrowstyle='<->', color=col, lw=3))
    ax.text(v[0]*1.1, v[1]*1.1, f'{lbl}\n({var_explained[i]*100:.0f}%)', color=col, fontsize=9, fontweight='bold')
ax.set_title('Principal component directions\n(arrow length ∝ variance captured)', fontweight='bold')
ax.set_xlabel('Feature 0'); ax.set_ylabel('Feature 1')
ax.set_aspect('equal')

# Right: variance explained bar chart
ax = axes[2]
bars = ax.bar(['PC1', 'PC2'], var_explained*100, color=['crimson','seagreen'], alpha=0.8, edgecolor='white', lw=1.5)
ax.set_ylabel('Variance explained (%)')
ax.set_title('Scree plot\nHow much information each PC captures', fontweight='bold')
for bar, val in zip(bars, var_explained*100):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{val:.1f}%',
            ha='center', fontsize=12, fontweight='bold')
ax.set_ylim(0, 110)

plt.suptitle('PCA: finding the directions that capture the most variance in your data',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()
print("PC1 direction captures most of the spread → you can project onto just PC1 and keep most information")

---
# Module 8 — Matrix Decompositions

## What Is a Decomposition?

A decomposition breaks a matrix into a product of simpler matrices. Think of it like factoring a number: 12 = 2 × 2 × 3. The factors reveal structure that the original number hides, and they make certain calculations much cheaper.

There are three decompositions you need to know:

## 8.1 LU Decomposition: A = L × U

- **L** = Lower triangular (zeros above the diagonal)
- **U** = Upper triangular (zeros below the diagonal)
- This is Gaussian elimination (from Module 5) stored as a matrix product

**Why it matters:** Solving AX = B from scratch takes O(n³) operations. After LU decomposition, solving LY = B and UX = Y each takes only O(n²). If you need to solve the same A with many different B vectors (e.g. in cross-validation), you do the expensive decomposition once and reuse it.

## 8.2 QR Decomposition: A = Q × R

- **Q** = Orthogonal matrix (all columns are unit vectors, all perpendicular to each other: QᵀQ = I)
- **R** = Upper triangular
- **Key property:** Q⁻¹ = Qᵀ — the inverse is just the transpose (trivial to compute!)

**Why it matters:** When you have more equations than unknowns (overdetermined systems, which is what happens in linear regression), you need least-squares. QR is the numerically stable way to solve this. `np.linalg.lstsq` uses QR internally.

## 8.3 SVD: A = U × Σ × Vᵀ (The Most Powerful)

SVD works on ANY matrix — square or not. Every matrix can be decomposed this way.

- **U** (m×m): left singular vectors — orthogonal matrix
- **Σ** (m×n): diagonal matrix containing the singular values σ₁ ≥ σ₂ ≥ ... ≥ 0
- **Vᵀ** (n×n): right singular vectors — orthogonal matrix

The **singular values** in Σ are like a "importance ranking" of each dimension. Large σᵢ = important direction. Near-zero σᵢ = almost redundant direction.

If you set the small singular values to zero and reconstruct, you get a compressed approximation of the original matrix — this is the basis of image compression, recommendation systems, and PCA.

In [ ]:
import numpy as np
from scipy.linalg import lu, svd as scipy_svd
import matplotlib.pyplot as plt

# =================================================================
# MODULE 8  LU, QR, SVD
# =================================================================

# ---- LU ----
A = np.array([[2.0,1,1],[4.0,3,3],[8.0,7,9]])
P, L, U = lu(A)
print("=== LU Decomposition ===")
print("L (lower triangular):"); print(L.round(3))
print("U (upper triangular):"); print(U.round(3))
print("P @ A == L @ U:", np.allclose(P @ A, L @ U))

# Solve for multiple b vectors using cached LU
from scipy.linalg import lu_factor, lu_solve
lu_obj = lu_factor(A)
for b in [np.array([4.,10.,20.]), np.array([1.,2.,3.])]:
    x = lu_solve(lu_obj, b)
    print(f"  b={b} → x={x.round(3)}")

# ---- QR ----
Q, R = np.linalg.qr(A)
print("\n=== QR Decomposition ===")
print("Q.T @ Q = I:", np.allclose(Q.T @ Q, np.eye(Q.shape[0])))
print("R (upper triangular):"); print(R.round(3))

# ---- SVD ----
A2 = np.array([[1.,2],[3,4],[5,6]])
U2, s, Vt = scipy_svd(A2)
print("\n=== SVD ===")
print("Singular values:", s.round(4))
print("Relative importance:", np.round(s**2/(s**2).sum()*100, 1), "%")

# Reconstruct from SVD
Sigma = np.zeros(A2.shape); Sigma[:len(s),:len(s)] = np.diag(s)
print("Reconstruction error:", np.max(np.abs(A2 - U2 @ Sigma @ Vt)))

# Truncated SVD: rank-1 approximation
Sig1 = np.zeros(A2.shape); Sig1[0,0] = s[0]
A_rank1 = U2 @ Sig1 @ Vt
print("\nOriginal vs Rank-1 approximation:")
print(A2)
print(A_rank1.round(3))
print(f"Keeps {s[0]**2/(s**2).sum()*100:.1f}% of information with rank-1")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import svd as scipy_svd

# =================================================================
# VISUALISATION: SVD — what singular values tell you
# =================================================================

# Create a "dataset" image to compress with SVD
np.random.seed(1)
n = 40
rank3_matrix = (np.outer(np.sin(np.linspace(0,np.pi,n)), np.cos(np.linspace(0,2*np.pi,n))) +
                0.5*np.outer(np.cos(np.linspace(0,2*np.pi,n)), np.sin(np.linspace(0,np.pi,n))) +
                0.1*np.random.randn(n,n))

U, s, Vt = scipy_svd(rank3_matrix)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Top row: original + reconstructions with increasing rank
for k, ax in zip([1, 2, 5, 40], axes[0]):
    Sig_k = np.zeros_like(rank3_matrix)
    Sig_k[:k,:k] = np.diag(s[:k])
    approx = U @ Sig_k @ Vt
    err = np.mean((rank3_matrix - approx)**2)
    ax.imshow(approx, cmap='viridis', aspect='auto', vmin=rank3_matrix.min(), vmax=rank3_matrix.max())
    ax.set_title(f'Top-{k} singular value{"s" if k>1 else ""}\nMSE={err:.4f}', fontweight='bold')
    ax.axis('off')
    if k == 40: ax.set_title('Original (rank-40)\nMSE=0', fontweight='bold')

# Bottom row: singular values and their importance
ax = axes[1][0]
ax.semilogy(range(1,len(s)+1), s, 'royalblue', marker='o', markersize=5, lw=2)
ax.set_xlabel('Singular value index'); ax.set_ylabel('Singular value (log scale)')
ax.set_title('Singular values decay rapidly\n→ early ones capture most info', fontweight='bold')
ax.axvline(5, color='crimson', lw=2, ls='--', label='keep top-5')
ax.legend()

ax = axes[1][1]
cumvar = (s**2).cumsum() / (s**2).sum() * 100
ax.plot(range(1,len(s)+1), cumvar, 'seagreen', marker='o', markersize=5, lw=2)
ax.axhline(95, color='crimson', ls='--', lw=2, label='95% threshold')
ax.axhline(99, color='orange', ls='--', lw=2, label='99% threshold')
ax.set_xlabel('Number of components kept'); ax.set_ylabel('Cumulative variance explained (%)')
ax.set_title('How many components do you need?', fontweight='bold')
ax.legend(fontsize=9)

ax = axes[1][2]
top5 = s[:5]**2/(s**2).sum()*100
ax.bar(range(1,6), top5, color='royalblue', alpha=0.8, edgecolor='white')
ax.set_xlabel('Singular value'); ax.set_ylabel('% variance explained')
ax.set_title(f'Top 5 components explain\n{top5.sum():.1f}% of variance', fontweight='bold')

ax = axes[1][3]
axes[1][3].axis('off')
axes[1][3].text(0.5, 0.5,
    f'40x40 matrix: {40*40} numbers.\n'
    f'Top-5 SVD: {5*(40+40+1)} numbers\n'
    f'({5*(40+40+1)/(40*40)*100:.0f}% of original).\n'
    'That is image compression.',
    ha='center', va='center', transform=axes[1][3].transAxes,
    fontsize=11, bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle('SVD: approximating a matrix with fewer components',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
# Module 9 — How Everything Connects in Real ML Pipelines

## Linear Regression: Tracing Every Piece

The linear regression formula is: **β = (XᵀX)⁻¹ Xᵀ y**

Let's trace each piece back to what we covered:

| Piece | Module | What it does |
|---|---|---|
| X | 2 | Your feature matrix (m samples × n features) |
| Xᵀ | 2.5 | Transpose — flips rows and columns to make dimensions align |
| XᵀX | 2.4 | Matrix multiply → produces an n×n square matrix |
| (XᵀX)⁻¹ | 4 | Inverse — only exists if X has full column rank (Module 6) |
| Xᵀy | 2.4 | Matrix-vector multiply — dot products of each feature column with the target |
| β | — | The coefficient vector: one number per feature, giving the best linear fit |

In practice, sklearn uses SVD internally instead of computing the explicit inverse (exactly as we discussed in Module 4 — never use inv()). But the formula above is the mathematical intent.

## Neural Networks: Every Layer Is a Matrix Multiply

A neural network layer with p neurons processing m samples:

```
Z = X @ Wᵀ + b

X:   shape (m × n)  — input batch (m samples, n input features)
Wᵀ:  shape (n × p)  — weight matrix transposed
b:   shape (p,)     — bias vector
Z:   shape (m × p)  — output (m samples, p neuron outputs)
```

- Each **neuron** computes a dot product of its weights with the input (Module 1.3)
- The **whole layer** is one matrix multiply (Module 2.4)
- **Backpropagation** is the chain rule applied to those matrix multiplies
- This is why you need a GPU — you are doing billions of matrix multiplies during training

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

# =================================================================
# MODULE 9  EVERYTHING COMES TOGETHER
# =================================================================

fleet = np.array([
    [8,  12.0, 3.5],
    [10, 15.0, 4.1],
    [12, 16.0, 4.4],
    [14, 20.0, 5.0],
    [16, 22.0, 5.3],
])
X_feat = fleet[:,:2]; y_fuel = fleet[:,2]

# --- Manual normal equation ---
X_design = np.hstack([np.ones((5,1)), X_feat])   # add bias column
beta = np.linalg.solve(X_design.T @ X_design, X_design.T @ y_fuel)
print("=== Linear Regression — Normal Equation ===")
print(f"β = [intercept, trips_coef, fare_coef] = {beta.round(4)}")

# sklearn (SVD internally)
lr = LinearRegression(); lr.fit(X_feat, y_fuel)
print(f"sklearn  = [{lr.intercept_:.4f}, {lr.coef_[0]:.4f}, {lr.coef_[1]:.4f}]")
print(f"Match? {np.allclose(beta, [lr.intercept_]+list(lr.coef_), atol=1e-6)}")

# --- Neural network forward pass ---
np.random.seed(0)
m, n_in, p = 5, 3, 4
X_batch = fleet.astype(float)
W = np.random.randn(p, n_in) * 0.5   # shape (4, 3)
b = np.zeros(p)                        # shape (4,)

Z = X_batch @ W.T + b      # shape (5, 4) — ONE matrix multiply for ALL neurons, ALL samples
A_relu = np.maximum(0, Z)  # ReLU activation

print("\n=== Neural Network Layer ===")
print(f"Input   X  : {X_batch.shape}   (5 matatus, 3 features)")
print(f"Weights W  : {W.shape}         (4 neurons, 3 weights each)")
print(f"Pre-act Z  : {Z.shape}   (5 matatus × 4 neurons) — ONE matrix multiply")
print(f"Post-ReLU  : {A_relu.shape}")
print("Z (pre-activation):")
print(Z.round(3))
print("After ReLU (negative values → 0):")
print(A_relu.round(3))
print("\nThis is one layer. A deep network stacks many of these in sequence.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =================================================================
# FINAL VISUALISATION: The entire linear algebra journey
# =================================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Vector (dot product)
ax = axes[0,0]
features = np.array([10.0, 145.0, 15.0])
weights  = np.array([0.5, 0.02, 1.0])
products = features * weights
ax.bar(['trips×0.5', 'pass×0.02', 'fare×1.0'], products,
       color=['royalblue','seagreen','crimson'], alpha=0.8, edgecolor='white')
ax.axhline(products.sum(), color='black', ls='--', lw=2, label=f'Total = {products.sum():.1f}')
ax.set_title(f'Module 1: Dot Product\n= prediction score = {products.sum():.1f}', fontweight='bold')
ax.legend(); ax.set_ylabel('Contribution')

# 2. Matrix (your dataset)
ax = axes[0,1]
fleet = np.array([[8,12,3.5],[10,15,4.1],[12,16,4.4],[14,20,5],[16,22,5.3]])
im = ax.imshow(fleet, cmap='Blues', aspect='auto')
ax.set_xticks([0,1,2]); ax.set_xticklabels(['trips','fare_k','fuel'])
ax.set_yticks(range(5)); ax.set_yticklabels([f'matatu {i+1}' for i in range(5)])
for i in range(5):
    for j in range(3):
        ax.text(j,i,fleet[i,j],ha='center',va='center',fontsize=10,fontweight='bold',
                color='white' if fleet[i,j]>10 else 'black')
ax.set_title('Module 2: Matrix\n= your entire dataset', fontweight='bold')

# 3. Determinant
ax = axes[0,2]
thetas = np.linspace(0,2*np.pi,300)
A_rot = np.array([[np.cos(0.5),-np.sin(0.5)],[np.sin(0.5),np.cos(0.5)]])*2
sq = np.array([[0,0],[1,0],[1,1],[0,1],[0,0]],dtype=float).T
tsq = A_rot @ sq
ax.fill(sq[0],sq[1],alpha=0.3,color='royalblue',label='unit square (area=1)')
ax.fill(tsq[0],tsq[1],alpha=0.3,color='crimson',label=f'transformed (area={abs(np.linalg.det(A_rot)):.0f})')
ax.plot(sq[0],sq[1],'royalblue',lw=2); ax.plot(tsq[0],tsq[1],'crimson',lw=2)
ax.axhline(0, color='k', lw=0.7); ax.axvline(0, color='k', lw=0.7)
ax.set_xlim(-3,3); ax.set_ylim(-3,3); ax.set_aspect('equal')
ax.set_title(f'Module 3: Determinant = {abs(np.linalg.det(A_rot)):.0f}\n(area scaling factor)', fontweight='bold')
ax.legend(fontsize=8)

# 4. Solving AX=B
ax = axes[1,0]
A = np.array([[2.,3],[1,-1]]); b = np.array([9.,3.])
x1_range = np.linspace(-1,6,200)
y1 = (b[0] - 2*x1_range)/3
y2 = x1_range - 3
ax.plot(x1_range,y1,'royalblue',lw=2.5,label='2x+3y=9')
ax.plot(x1_range,y2,'crimson',lw=2.5,label='x-y=3')
sol = np.linalg.solve(A,b)
ax.scatter(*sol,s=200,color='gold',zorder=5,label=f'Solution ({sol[0]:.1f},{sol[1]:.1f})',edgecolors='black',lw=1.5)
ax.set_xlim(-1,6); ax.set_ylim(-2,5)
ax.set_title('Module 5: Solving AX=B\nIntersection = solution', fontweight='bold')
ax.legend(fontsize=9); ax.set_xlabel('x'); ax.set_ylabel('y')

# 5. Eigenvalues / PCA
ax = axes[1,1]
np.random.seed(7)
t = np.random.randn(60)
d = np.array([np.cos(np.pi/4),np.sin(np.pi/4)])
data = np.outer(t,d) + np.random.randn(60,2)*0.25
ax.scatter(data[:,0],data[:,1],alpha=0.5,color='royalblue',s=40)
C = np.cov(data.T); vals,vecs = np.linalg.eigh(C)
idx = np.argsort(vals)[::-1]; vals,vecs = vals[idx],vecs[:,idx]
for i,col in enumerate(['crimson','seagreen']):
    v = vecs[:,i]*np.sqrt(vals[i])*2
    ax.annotate('',xy=v,xytext=-v,arrowprops=dict(arrowstyle='<->',color=col,lw=3))
    ax.text(v[0]*1.1,v[1]*1.1,f'PC{i+1}\n({vals[i]/vals.sum()*100:.0f}%)',color=col,fontsize=9,fontweight='bold')
ax.set_aspect('equal'); ax.set_title('Module 7: Eigenvalues/PCA\nPrincipal component directions', fontweight='bold')

# 6. SVD compression
ax = axes[1,2]
np.random.seed(1)
M = np.random.randn(30,30)
M += 5*np.outer(np.sin(np.linspace(0,np.pi,30)),np.cos(np.linspace(0,2*np.pi,30)))
U,s,Vt = np.linalg.svd(M)
ks = [1,2,5,10,30]; errors = []
for k in ks:
    Mk = (U[:,:k]*s[:k]) @ Vt[:k,:]
    errors.append(np.mean((M-Mk)**2))
ax.semilogy(ks,errors,'royalblue',marker='o',markersize=8,lw=2.5)
ax.set_xlabel('Number of singular values kept')
ax.set_ylabel('Reconstruction error (log)')
ax.set_title('Module 8: SVD Compression\nMore singular values → better approximation', fontweight='bold')
ax.axvline(5,color='crimson',ls='--',lw=2,label='Good trade-off'); ax.legend()

plt.suptitle('Linear Algebra in ML — The Complete Picture', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

---
# Quick Reference — Everything in One Table

| What you want to do | Tool | numpy / sklearn code |
|---|---|---|
| Score a single input against weights | Dot product | `np.dot(features, weights)` |
| Compare patterns between observations | Cosine similarity | `cosine_similarity(X)` |
| Apply a transformation / make predictions for all samples | Matrix multiply | `A @ B` |
| Flip rows and columns | Transpose | `A.T` |
| Get all feature relationships at once | Covariance matrix | `np.cov(X.T)` |
| Get all Pearson r values at once | Correlation matrix | `df.corr()` |
| Check if a matrix can be inverted | Determinant | `np.linalg.det(A)` |
| Check for redundant features | Rank | `np.linalg.matrix_rank(A)` |
| Find features that are invisible to a model | Null space | `scipy.linalg.null_space(A)` |
| Solve AX = B (ALWAYS use this, not inv) | Direct solve | `np.linalg.solve(A, b)` |
| Solve regression (more data points than unknowns) | Least squares | `np.linalg.lstsq(X, y)` |
| Find principal component directions | Eigendecomposition | `np.linalg.eig(A)` |
| Reduce dimensions | PCA | `PCA(n_components=k).fit_transform(X)` |
| Decompose any matrix (compress, denoise, recommend) | SVD | `np.linalg.svd(A)` |
| Repeated solves, same A | LU decomposition | `scipy.linalg.lu_factor(A)` then `lu_solve` |

---

> **The one thing to take away:**  
> Every time a machine learning algorithm processes a batch of data — making predictions,  
> updating weights, computing similarities — it is doing **matrix multiplication**.  
> All the concepts in this notebook are either building up to that operation or  
> revealing the structure hidden inside it.